In [0]:
access_key = ''
secret_key = ''.replace('/', '%2F')
aws_bucket_name = ''

csv_path = f"s3a://{access_key}:{secret_key}@{aws_bucket_name}/credit_card_pyspark/csv"
json_path = f"s3a://{access_key}:{secret_key}@{aws_bucket_name}/credit_card_pyspark/json"
parquet_path = f"s3a://{access_key}:{secret_key}@{aws_bucket_name}/credit_card_pyspark/parquet"

In [0]:
from pyspark.sql.functions import col, to_date, year
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema = StructType([
    StructField("Transaction_ID", StringType(), True),
    StructField("Transaction_Date", StringType(), True),
    StructField("Transaction_Amount", DoubleType(), True),
    StructField("Transaction_Status", StringType(), True),
    StructField("Transaction_Type", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Name", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("DOB", StringType(), True),
    StructField("Email", StringType(), True),
    StructField("Phone", StringType(), True),
    StructField("Customer_City", StringType(), True),
    StructField("Card_ID", StringType(), True),
    StructField("Card_Type", StringType(), True),
    StructField("Issuer_Bank", StringType(), True),
    StructField("Card_Tier", StringType(), True),
    StructField("Expiry_Date", StringType(), True),
    StructField("Merchant_ID", StringType(), True),
    StructField("Merchant_Name", StringType(), True),
    StructField("Merchant_Category", StringType(), True),
    StructField("Merchant_Country", StringType(), True),
    StructField("Location_ID", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Country", StringType(), True)
])

In [0]:
csv_df = spark.read.option("header", True).schema(schema).csv(csv_path)
display(csv_df)


In [0]:
json_df = spark.read.option('multiline', True).json(json_path)
display(json_df)

In [0]:
parquet_df = spark.read.schema(schema).parquet(parquet_path)
display(parquet_df)

In [0]:
schema_cols = [
    "Transaction_ID", "Customer_ID", "Card_ID", "Merchant_ID", "Location_ID",
    "Transaction_Date", "DOB", "Expiry_Date",
    "Customer_Name", "Gender", "Email", "Phone", "Customer_City",
    "Card_Type", "Issuer_Bank", "Card_Tier",
    "Transaction_Status", "Transaction_Type", "Transaction_Amount",
    "Merchant_Name", "Merchant_Category", "Merchant_Country",
    "City", "State", "Country"
]

def align(df):
    return df.select([col(c) for c in schema_cols])


In [0]:
csv_df = align(csv_df)
display(csv_df)

In [0]:
json_df = align(json_df)
display(json_df)

In [0]:
parquet_df = align(parquet_df)
display(parquet_df)

In [0]:
combined_df = csv_df.union(json_df).union(parquet_df)
display(combined_df)

In [0]:
combined_df = combined_df.withColumn('year', year(to_date(col('Transaction_Date'))))


In [0]:
par_df = combined_df.repartition("year")
par_df.write.partitionBy("year").mode("overwrite").saveAsTable('prod.bronze_raw.transactions')